# ML Pipeline Contracts

Two synthetic domains, one Model Scope schema, one set of five stage contracts. This
notebook reads only what `src/orchestrator.py` already produced under `runs/`, the same
convention project 08 uses for its notebook: no detector, no model, no stage function
gets called again here, every chart below comes from a committed artifact.


In [1]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

import sys
sys.path.insert(0, str(Path.cwd().parent / "src"))
from style import set_style, style_ax, savefig, MUTED_AMBER, MUTED_RED, MUTED_TEAL, SLATE, GREY

BASE = Path.cwd().parent
FIG_DIR = BASE / "reports" / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)
set_style()


def load_run(slug):
    run_dir = BASE / "runs" / slug / "run-001"
    art = run_dir / "artifacts"
    return {
        "target_exploration": json.loads((art / "eda" / "target_exploration.json").read_text()),
        "feature_importance": pd.read_csv(art / "model" / "feature_importance.csv"),
        "reduction_history": json.loads((art / "model" / "reduction_history.json").read_text()),
        "validation_report": (art / "validation" / "report.md").read_text(),
    }


churn = load_run("churn")
ticket = load_run("ticket-triage")
print("loaded churn and ticket-triage run artifacts")


loaded churn and ticket-triage run artifacts


## 1. Domain A: subscription churn

`churned_next_30d`, a single classification target, decided at each customer's 90-day
mark from usage behavior in the 90 days before it.


In [2]:
rate_by_month = churn["target_exploration"]["churned_next_30d"]["rate_by_month"]
months = list(rate_by_month.keys())
rates = [v * 100 for v in rate_by_month.values()]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(months, rates, marker="o", color=SLATE, linewidth=1.8)
style_ax(ax, title="Churn rate sits in the high-20s outside a thin first month",
         subtitle="churned_next_30d rate, by decision-date month (2024-03 has only 12 decisions, everything after has 200+)",
         ylabel="Churn rate (%)")
ax.tick_params(axis="x", rotation=45)
savefig(fig, FIG_DIR / "churn_rate_by_month.png", footnote="Source: runs/churn/run-001/artifacts/eda/target_exploration.json, data/churn/customers.csv")
plt.show()


In [3]:
fi = churn["feature_importance"].sort_values("gain", ascending=True)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(fi["feature"], fi["gain"], color=MUTED_TEAL, zorder=3)
style_ax(ax, title="Usage behavior dominates the churn model's selected features",
         subtitle="Gain-based importance, after 5-step reduction",
         xlabel="Gain")
savefig(fig, FIG_DIR / "churn_feature_importance.png", footnote="Source: runs/churn/run-001/artifacts/model/feature_importance.csv")
plt.show()
churn["feature_importance"]


,target,feature,gain,iv,psi_train_vs_holdout
0,churned_next_30d,plan_tier_basic,0.13897,0.14260,0.00000
1,churned_next_30d,tickets_opened_pre_decision,0.20620,0.10392,0.00604
2,churned_next_30d,plan_tier_enterprise,0.02667,0.08005,0.00000
3,churned_next_30d,avg_usage_score_pre_decision,0.33406,0.01847,0.01828
4,churned_next_30d,n_events_pre_decision,0.08411,0.02902,0.02151
5,churned_next_30d,avg_logins_pre_decision,0.21000,0.00959,0.01554


In [4]:
steps = churn["reduction_history"]["churned_next_30d"]["steps"]
n_feat = [s["n_features"] for s in steps]
metric = [s["metric"] for s in steps]

fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(n_feat, metric, marker="o", color=MUTED_AMBER, linewidth=1.8)
best_idx = max(range(len(metric)), key=lambda i: metric[i])
ax.scatter([n_feat[best_idx]], [metric[best_idx]], color=MUTED_RED, zorder=5, s=70, label="kept")
style_ax(ax, title="Dropping the weakest features doesn't cost PR-AUC here",
         subtitle="5-step iterative reduction, weakest feature dropped each round; the kept step is the best holdout score seen",
         xlabel="Features remaining", ylabel="PR-AUC (holdout)")
ax.invert_xaxis()
ax.legend()
savefig(fig, FIG_DIR / "churn_reduction_funnel.png", footnote="Source: runs/churn/run-001/artifacts/model/reduction_history.json")
plt.show()


## 2. Domain B: support ticket triage

Two targets from the same base population, `will_escalate` (classification) and
`resolution_hours` (regression), both decided at the 2-hour triage mark. Same Model
Scope schema, same five stage contracts, same orchestrator code as domain A, this is
the actual point of running two domains: nothing about the pipeline changed to handle
`task_type: both`.


In [5]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

esc_rate = ticket["target_exploration"]["will_escalate"]["rate_by_month"]
axes[0].plot(list(esc_rate.keys()), [v * 100 for v in esc_rate.values()], marker="o", color=SLATE, linewidth=1.8)
style_ax(axes[0], title="will_escalate", subtitle="Escalation rate by month", ylabel="Escalation rate (%)")
axes[0].tick_params(axis="x", rotation=45)

res_mean = ticket["target_exploration"]["resolution_hours"]["mean_by_month"]
axes[1].plot(list(res_mean.keys()), list(res_mean.values()), marker="o", color=MUTED_TEAL, linewidth=1.8)
style_ax(axes[1], title="resolution_hours", subtitle="Mean resolution time by month", ylabel="Hours")
axes[1].tick_params(axis="x", rotation=45)

fig.tight_layout()
savefig(fig, FIG_DIR / "ticket_targets_by_month.png", footnote="Source: runs/ticket-triage/run-001/artifacts/eda/target_exploration.json")
plt.show()


In [6]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
for ax, target, color in zip(axes, ["will_escalate", "resolution_hours"], [MUTED_TEAL, MUTED_AMBER]):
    steps = ticket["reduction_history"][target]["steps"]
    n_feat = [s["n_features"] for s in steps]
    metric = [s["metric"] for s in steps]
    ax.plot(n_feat, metric, marker="o", color=color, linewidth=1.8)
    metric_name = ticket["reduction_history"][target]["metric_name"]
    style_ax(ax, title=target, subtitle=f"Reduction funnel ({metric_name})", xlabel="Features remaining", ylabel=metric_name)
    ax.invert_xaxis()
fig.tight_layout()
savefig(fig, FIG_DIR / "ticket_reduction_funnels.png", footnote="Source: runs/ticket-triage/run-001/artifacts/model/reduction_history.json")
plt.show()


## 3. Same contract, three targets, one validation gate

Three targets across the two domains, each checked against the threshold its own Model
Scope declared, not a shared default.


In [7]:
print(churn["validation_report"])
print(ticket["validation_report"])


# Validation report

## churned_next_30d

- pr_auc on holdout: 0.3918
- threshold: 0.3 (goal: maximize)
- **PASS**

## Overall

**PASS** — 1/1 targets clear their threshold.

# Validation report

## will_escalate

- pr_auc on holdout: 0.4353
- threshold: 0.4 (goal: maximize)
- **PASS**

## resolution_hours

- mae on holdout: 3.0471
- threshold: 4.5 (goal: minimize)
- **PASS**

## Overall

**PASS** — 2/2 targets clear their threshold.



In [8]:
import yaml

summary = []
for slug in ["churn", "ticket-triage"]:
    header = yaml.safe_load((BASE / "runs" / slug / "run-001" / "MODEL_SCOPE.md").read_text().split("---")[1])
    hist = json.loads((BASE / "runs" / slug / "run-001" / "artifacts" / "model" / "reduction_history.json").read_text())
    for metric_spec in header["metrics"]:
        target = metric_spec["target"]
        steps = hist[target]["steps"]
        # the selected step is whichever one the reduction loop kept, the
        # best holdout metric seen, not necessarily the most-reduced one
        best_step = max(steps, key=lambda s: s["metric"]) if metric_spec["goal"] == "maximize" \
            else min(steps, key=lambda s: s["metric"])
        summary.append({
            "domain": slug, "target": target, "metric": metric_spec["name"],
            "achieved": best_step["metric"], "threshold": metric_spec["threshold"], "goal": metric_spec["goal"],
        })

summary_df = pd.DataFrame(summary)
summary_df["passed"] = summary_df.apply(
    lambda r: r["achieved"] >= r["threshold"] if r["goal"] == "maximize" else r["achieved"] <= r["threshold"], axis=1)
summary_df


,domain,target,metric,achieved,threshold,goal,passed
0,churn,churned_next_30d,pr_auc,0.39179,0.3,maximize,True
1,ticket-triage,will_escalate,pr_auc,0.43528,0.4,maximize,True
2,ticket-triage,resolution_hours,mae,3.04708,4.5,minimize,True


In [9]:
fig, ax = plt.subplots(figsize=(8.5, 4.5))
labels = [f"{r.domain}\n{r.target}\n({r.metric})" for r in summary_df.itertuples()]
colors = [MUTED_TEAL if p else MUTED_RED for p in summary_df["passed"]]
ax.bar(labels, summary_df["achieved"], color=colors, zorder=3, width=0.5, label="achieved")
ax.scatter(labels, summary_df["threshold"], color=GREY, zorder=4, marker="_", s=900, linewidth=2.5, label="threshold")
style_ax(ax, title="Three targets, two domains, one validation gate each",
         subtitle="Achieved metric vs. each target's own Model Scope threshold")
ax.legend()
savefig(fig, FIG_DIR / "cross_domain_validation_summary.png",
        footnote="Source: runs/{churn,ticket-triage}/run-001/artifacts/validation/, MODEL_SCOPE.md")
plt.show()


## 4. What the contract layer catches

`src/demo_broken_contract.py` runs EDA and Data for real against the churn Model
Scope, then simulates the single most common way a Feature Engineering stage breaks
its handoff to Training: the target rides along as a declared feature. Real captured
output, not paraphrased.


In [10]:
print((BASE / "runs" / "broken-example" / "run-001" / "CAUGHT_VIOLATION.md").read_text())


# Caught contract violation (real output, not paraphrased)

Stages 1-2 (EDA, Data) ran for real against `runs/churn/run-001`'s Model Scope and
produced a valid `training_data.parquet`. Stage 3 (Feature Engineering) is simulated
here as broken: its declared `feature_cols` include `churned_next_30d`, the target
itself, the single most common way a feature matrix leaks its own label.

```
ContractViolation: [features] target column(s) ['churned_next_30d'] appear in feature_cols, that's target leakage
```

The orchestrator stops here: Training never runs against this feature matrix.



## Takeaways

The same `ModelScope` schema, the same five `verify_*` contract checks, and the same
orchestrator loop ran a single-target classification domain and a two-target
`task_type: both` domain without a single change to `src/orchestrator.py` or
`schemas/`. The only things that changed between domains were the two files that were
always supposed to change: `bindings.yaml` (execution detail) and `feature_spec.yaml`
(the DS-authored feature ideas). The one deliberately broken run shows what the
contract layer is actually for: catching a bad handoff between stages before it
becomes a silently-wrong model, not after.
